# Arabic SFT & DPO Pipeline

**Production-Grade Arabic Training Data Generation**

✅ Saudi Arabic Dialect (95+ expressions)  
✅ Bilingual Thinking (Arabic + English with terms preserved)  
✅ Structured Q/T/A Templates  
✅ 15+ Quality Metrics  
✅ Error Resilience  
✅ 3000+ DPO Pairs  

**Pipeline:** Setup → Load SFT → Validate → Accept → Generate 3000+ DPO → Export

## 1. Setup & Environment

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from getpass import getpass

# Initialize project path (portable - works on any computer)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Fallback: go up one level if src not in current directory
    PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'src').exists():
        raise FileNotFoundError('Could not find src/ directory. Make sure you run this notebook from the project root.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Import modules
try:
    from sft import (
        SFTGenerator,
        SFTValidator,
        QualityScorer,
        load_chunks,
        save_samples,
        DPOPrompts,
    )
    print('✓ All SFT modules imported successfully')
except ImportError as e:
    print(f'✗ Import error: {e}')
    raise

print(f'✓ Project root: {PROJECT_ROOT}')

In [ ]:
# Load environment variablesdef load_dotenv(path):    if not path.exists():        return    for raw_line in path.read_text(encoding='utf-8').splitlines():        line = raw_line.strip()        if not line or line.startswith('#') or '=' not in line:            continue        name, value = line.split('=', 1)        name, value = name.strip(), value.strip().strip('"').strip("'")        if name and value:            os.environ.setdefault(name, value)load_dotenv(PROJECT_ROOT / '.env')OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')if not OPENROUTER_API_KEY:    OPENROUTER_API_KEY = getpass('OpenRouter API key: ').strip()    if OPENROUTER_API_KEY:        os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEYMODEL = os.environ.get('OPENROUTER_MODEL', 'google/gemini-2.5-flash-lite')print(f'✓ API Key configured')print(f'✓ Model: {MODEL}')

In [ ]:
# Configure paths and settings
CHUNKS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'chunks_asas_albalagha.jsonl'
GENERATED_DIR = PROJECT_ROOT / 'data' / 'generated'
SFT_CANDIDATES_OUTPUT = GENERATED_DIR / 'sft' / 'candidates.jsonl'
SFT_ACCEPTED_OUTPUT = GENERATED_DIR / 'sft' / 'accepted.jsonl'
DPO_CANDIDATES_OUTPUT = GENERATED_DIR / 'dpo' / 'candidates.jsonl'

# DPO Generation settings
DPO_TARGET = 3000               # Target 3000+ DPO pairs
PAUSE_BETWEEN_DPO_CALLS = 0.5   # API call delay

print('\n=== CONFIGURATION ===')
print(f'Project: {PROJECT_ROOT.name}')
print(f'\nSettings:')
print(f'  DPO target: {DPO_TARGET}+ pairs')
print(f'  API delay: {PAUSE_BETWEEN_DPO_CALLS}s')

## 2. Load SFT Candidates

In [ ]:
print('\n=== LOADING SFT DATA ===')

try:
    from sft.schema import SFTSample
    sft_samples = []
    
    if SFT_CANDIDATES_OUTPUT.exists():
        with open(SFT_CANDIDATES_OUTPUT, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data = json.loads(line)
                    sample = SFTSample.from_dict(data)
                    sft_samples.append(sample)
        
        print(f'✓ Loaded {len(sft_samples):,} SFT candidates')
    else:
        print(f'✗ Candidates file not found: {SFT_CANDIDATES_OUTPUT}')
        sft_samples = []
except Exception as e:
    print(f'✗ Error loading candidates: {e}')
    sft_samples = []

## 3. Validate Quality

In [ ]:
print('\n=== VALIDATION ===')

try:
    is_valid, errors = SFTValidator.validate_contract(sft_samples)
    if is_valid:
        print(f'✓ Schema validation PASSED for {len(sft_samples):,} samples')
    else:
        print(f'✗ Schema validation FAILED:')
        for err in errors[:5]:
            print(f'  {err}')
except Exception as e:
    print(f'✗ Validation error: {e}')

## 4. Accept & Filter Quality Samples

In [ ]:
if sft_samples:
    accepted = []
    
    for sample in sft_samples:
        try:
            scores = QualityScorer.score_sample(sample)
            conf = scores.get('teacher_confidence', 0.0)
            if conf >= 0.7:
                accepted.append(sample)
        except Exception as e:
            continue
    
    rate = len(accepted) / len(sft_samples) * 100 if sft_samples else 0
    print(f'\n=== ACCEPTANCE ===')
    print(f'✓ Accepted {len(accepted):,} sample(s) ({rate:.1f}%)')
    
    if accepted:
        try:
            SFT_ACCEPTED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
            save_samples(accepted, str(SFT_ACCEPTED_OUTPUT))
            print(f'✓ Saved accepted samples to:')
            print(f'  {SFT_ACCEPTED_OUTPUT}')
        except Exception as e:
            print(f'✗ Error saving: {e}')
else:
    accepted = []
    print('⚠️  No samples to filter')

## 5. Generate 3000+ DPO Pairs

In [ ]:
DPO_TYPES = (
    'partial_factual_errors',
    'less_faithful_reconstruction',
    'unsupported_additions',
    'missing_information',
    'wrong_register',
    'weak_organization',
    'poor_instruction_following',
    'wrong_formatting',
    'verbosity',
)

if not OPENROUTER_API_KEY:
    print('⚠️  DPO paused: no API key')
    dpo_candidates = []
elif not accepted:
    print('⚠️  DPO paused: no accepted samples')
    dpo_candidates = []
else:
    print(f'\n=== GENERATING 3000+ DPO PAIRS (SCAFFOLD-FREE) ===')
    print(f'Target: {DPO_TARGET}+ pairs from {len(accepted):,} accepted samples')
    print(f'Method: Extract answer without thinking scaffold\n')
    
    from sft.generator import extract_answer_without_scaffold
    
    generator = SFTGenerator(OPENROUTER_API_KEY, MODEL)
    dpo_candidates = []
    pair_counter = 0
    
    # Calculate pairs needed per sample to reach target
    pairs_per_sample = max(1, (DPO_TARGET + len(accepted) - 1) // len(accepted))
    print(f'Strategy: Generate ~{pairs_per_sample} pair(s) per sample\n')
    
    for idx, sft in enumerate(accepted, 1):
        if pair_counter >= DPO_TARGET:
            print(f'\n✓ Reached target: {pair_counter:,} DPO pairs')
            break
        
        generated_for_sample = 0
        
        # Extract clean chosen (without thinking scaffold)
        inst = sft.messages[0].content
        full_response = sft.messages[1].content
        clean_chosen = extract_answer_without_scaffold(full_response)
        
        # Generate multiple rejection types per sample
        for rej_idx, rej_type in enumerate(DPO_TYPES[:pairs_per_sample]):
            if pair_counter >= DPO_TARGET:
                break
            
            try:
                msgs = DPOPrompts.build_messages(inst, clean_chosen, [rej_type])
                content, _ = generator.client.call(msgs)
                
                resp = json.loads(content)
                rej = resp.get('rejected', '').strip()
                
                if rej:
                    dpo_candidates.append({
                        'pair_id': f'{sft.sample_id}_dpo_{rej_idx:02d}',
                        'source_sample_id': sft.sample_id,
                        'source_chunk_id': sft.chunk_id,
                        'prompt': [{'role': 'user', 'content': inst}],
                        'chosen': [{'role': 'assistant', 'content': clean_chosen}],
                        'rejected': [{'role': 'assistant', 'content': rej}],
                        'rejection_type': rej_type,
                        'verification_status': 'Unverified',
                    })
                    pair_counter += 1
                    generated_for_sample += 1
                    
            except Exception as e:
                continue
            
            time.sleep(PAUSE_BETWEEN_DPO_CALLS)
        
        if idx % 50 == 0:
            print(f'  [{idx:5d}/{len(accepted):5d}] Generated {pair_counter:,} pairs so far')
    
    print(f'\n✓ Generated {len(dpo_candidates):,} DPO pairs (scaffold-free)')

## 6. Save DPO Candidates

In [ ]:
if dpo_candidates:
    try:
        DPO_CANDIDATES_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        with open(DPO_CANDIDATES_OUTPUT, 'w', encoding='utf-8') as f:
            for pair in dpo_candidates:
                f.write(json.dumps(pair, ensure_ascii=False) + '\n')
        print(f'\n=== EXPORT ===')
        print(f'✓ Saved {len(dpo_candidates):,} DPO pairs')
        print(f'  Location: {DPO_CANDIDATES_OUTPUT}')
        print(f'  File size: {DPO_CANDIDATES_OUTPUT.stat().st_size / (1024*1024):.1f} MB')
    except Exception as e:
        print(f'✗ Error saving: {e}')
else:
    print('⚠️  No DPO pairs to save')

## 7. Final Summary

In [ ]:
print('\n' + '='*60)
print('PIPELINE COMPLETE')
print('='*60)
print(f'SFT Candidates:   {len(sft_samples):>10,}')
print(f'SFT Accepted:     {len(accepted):>10,}')
if sft_samples:
    print(f'  Acceptance rate: {len(accepted)/len(sft_samples)*100:>10.1f}%')
print(f'DPO Pairs:        {len(dpo_candidates):>10,}')
if accepted:
    print(f'  Pairs per SFT:   {len(dpo_candidates)/len(accepted):>10.2f}x')
print('='*60)
print('\n✓ Arabic SFT/DPO dataset ready for training!')